> **Weather-aware workflow note:** Weather plots and correlations are exploratory analysis; weather becomes a forecasting input in the feature-engineering and model notebooks.

How does solar generation vary during the day?
How does irradiation affect AC/DC power?
Which plant produces more?
Which inverters perform differently?
How does temperature affect generation?
Are there unusual operating periods?
What are the daily/weekly generation patterns?

In [ ]:
# Import libraries required for exploratory data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define the location of cleaned datasets
PROCESSED_PATH = "../data/processed/"

# Load cleaned datasets
plant1_generation = pd.read_csv(
    PROCESSED_PATH + "plant1_generation_clean.csv"
)

plant1_weather = pd.read_csv(
    PROCESSED_PATH + "plant1_weather_clean.csv"
)

plant2_generation = pd.read_csv(
    PROCESSED_PATH + "plant2_generation_clean.csv"
)

plant2_weather = pd.read_csv(
    PROCESSED_PATH + "plant2_weather_clean.csv"
)

In [ ]:
# Convert DATE_TIME back to datetime after reading CSV files
for df in [
    plant1_generation,
    plant1_weather,
    plant2_generation,
    plant2_weather
]:
    df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])

In [ ]:
# Display the size of each cleaned dataset
print("Plant 1 Generation:", plant1_generation.shape)
print("Plant 1 Weather:", plant1_weather.shape)
print("Plant 2 Generation:", plant2_generation.shape)
print("Plant 2 Weather:", plant2_weather.shape)

In [ ]:
# Compare important generation statistics between both plants
print("Plant 1 Generation Statistics")
display(
    plant1_generation[
        ["DC_POWER", "AC_POWER", "DAILY_YIELD"]
    ].describe()
)

print("\nPlant 2 Generation Statistics")
display(
    plant2_generation[
        ["DC_POWER", "AC_POWER", "DAILY_YIELD"]
    ].describe()
)

In [ ]:
# Plot DC and AC power distributions for Plant 1
plt.figure(figsize=(10, 5))

plt.hist(
    plant1_generation["DC_POWER"],
    bins=50,
    alpha=0.6,
    label="DC Power"
)

plt.hist(
    plant1_generation["AC_POWER"],
    bins=50,
    alpha=0.6,
    label="AC Power"
)

plt.xlabel("Power")
plt.ylabel("Frequency")
plt.title("Plant 1 Power Distribution")
plt.legend()
plt.show()

In [ ]:
# Plot DC and AC power distributions for Plant 2
plt.figure(figsize=(10, 5))

plt.hist(
    plant2_generation["DC_POWER"],
    bins=50,
    alpha=0.6,
    label="DC Power"
)

plt.hist(
    plant2_generation["AC_POWER"],
    bins=50,
    alpha=0.6,
    label="AC Power"
)

plt.xlabel("Power")
plt.ylabel("Frequency")
plt.title("Plant 2 Power Distribution")
plt.legend()
plt.show()

In [ ]:
# Compare solar irradiation distributions
plt.figure(figsize=(10, 5))

plt.hist(
    plant1_weather["IRRADIATION"],
    bins=50,
    alpha=0.6,
    label="Plant 1"
)

plt.hist(
    plant2_weather["IRRADIATION"],
    bins=50,
    alpha=0.6,
    label="Plant 2"
)

plt.xlabel("Irradiation")
plt.ylabel("Frequency")
plt.title("Solar Irradiation Distribution")
plt.legend()
plt.show()

In [ ]:
# Compare ambient and module temperature for Plant 1
plt.figure(figsize=(10, 5))

plt.plot(
    plant1_weather["DATE_TIME"],
    plant1_weather["AMBIENT_TEMPERATURE"],
    label="Ambient Temperature"
)

plt.plot(
    plant1_weather["DATE_TIME"],
    plant1_weather["MODULE_TEMPERATURE"],
    label="Module Temperature"
)

plt.xlabel("Date")
plt.ylabel("Temperature")
plt.title("Plant 1 Temperature Over Time")
plt.legend()
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Aggregate inverter-level AC power to plant-level power
plant1_power = (
    plant1_generation
    .groupby("DATE_TIME")["AC_POWER"]
    .sum()
    .reset_index()
)

plant2_power = (
    plant2_generation
    .groupby("DATE_TIME")["AC_POWER"]
    .sum()
    .reset_index()
)

In [ ]:
# Merge plant-level power with weather observations
plant1_analysis = plant1_power.merge(
    plant1_weather[
        ["DATE_TIME", "IRRADIATION", "AMBIENT_TEMPERATURE",
         "MODULE_TEMPERATURE"]
    ],
    on="DATE_TIME",
    how="inner"
)

plant2_analysis = plant2_power.merge(
    plant2_weather[
        ["DATE_TIME", "IRRADIATION", "AMBIENT_TEMPERATURE",
         "MODULE_TEMPERATURE"]
    ],
    on="DATE_TIME",
    how="inner"
)

In [ ]:
# Visualize relationship between irradiation and AC power
plt.figure(figsize=(10, 6))

plt.scatter(
    plant1_analysis["IRRADIATION"],
    plant1_analysis["AC_POWER"],
    alpha=0.3
)

plt.xlabel("Irradiation")
plt.ylabel("AC Power")
plt.title("Plant 1: Irradiation vs AC Power")
plt.show()

In [ ]:
# Visualize Plant 2 irradiation and AC power relationship
plt.figure(figsize=(10, 6))

plt.scatter(
    plant2_analysis["IRRADIATION"],
    plant2_analysis["AC_POWER"],
    alpha=0.3
)

plt.xlabel("Irradiation")
plt.ylabel("AC Power")
plt.title("Plant 2: Irradiation vs AC Power")
plt.show()

Why this matters
Conceptually we expect:
Higher irradiation
       ↓
More solar energy
       ↓
Higher DC power
       ↓
Higher AC power

But we will let your actual dataset show the relationship, rather than assuming it.

In [ ]:
# Step 14: Analyze average solar generation by hour of day
# This helps us understand the daily solar generation pattern.

# Create hour feature from timestamp
plant1_analysis["HOUR"] = plant1_analysis["DATE_TIME"].dt.hour
plant2_analysis["HOUR"] = plant2_analysis["DATE_TIME"].dt.hour

# Calculate average AC power for each hour
plant1_hourly = (
    plant1_analysis
    .groupby("HOUR")["AC_POWER"]
    .mean()
)

plant2_hourly = (
    plant2_analysis
    .groupby("HOUR")["AC_POWER"]
    .mean()
)

print("Plant 1 Average Power by Hour:")
print(plant1_hourly)

print("\nPlant 2 Average Power by Hour:")
print(plant2_hourly)

In [ ]:
# Plot average solar generation throughout the day

plt.figure(figsize=(10, 5))

plt.plot(
    plant1_hourly.index,
    plant1_hourly.values,
    marker="o",
    label="Plant 1"
)

plt.plot(
    plant2_hourly.index,
    plant2_hourly.values,
    marker="o",
    label="Plant 2"
)

plt.xlabel("Hour of Day")
plt.ylabel("Average AC Power")
plt.title("Average Solar Power Generation by Hour")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Step 16: Calculate daily average AC power
# This helps identify daily generation patterns and unusual days.

plant1_analysis["DATE"] = plant1_analysis["DATE_TIME"].dt.date
plant2_analysis["DATE"] = plant2_analysis["DATE_TIME"].dt.date

plant1_daily = (
    plant1_analysis
    .groupby("DATE")["AC_POWER"]
    .mean()
)

plant2_daily = (
    plant2_analysis
    .groupby("DATE")["AC_POWER"]
    .mean()
)

print("Plant 1 Daily Power:")
print(plant1_daily.head())

print("\nPlant 2 Daily Power:")
print(plant2_daily.head())

In [ ]:
# Plot daily average generation

plt.figure(figsize=(12, 5))

plt.plot(
    plant1_daily.index,
    plant1_daily.values,
    label="Plant 1"
)

plt.plot(
    plant2_daily.index,
    plant2_daily.values,
    label="Plant 2"
)

plt.xlabel("Date")
plt.ylabel("Average AC Power")
plt.title("Daily Solar Generation Pattern")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Step 17: Analyze relationship between temperature and AC power

plt.figure(figsize=(10, 6))

plt.scatter(
    plant1_analysis["MODULE_TEMPERATURE"],
    plant1_analysis["AC_POWER"],
    alpha=0.3
)

plt.xlabel("Module Temperature")
plt.ylabel("AC Power")
plt.title("Plant 1: Module Temperature vs AC Power")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    plant2_analysis["MODULE_TEMPERATURE"],
    plant2_analysis["AC_POWER"],
    alpha=0.3
)

plt.xlabel("Module Temperature")
plt.ylabel("AC Power")
plt.title("Plant 2: Module Temperature vs AC Power")
plt.show()

In [ ]:
# Step 18: Calculate correlations between weather variables and AC power

plant1_corr = plant1_analysis[
    [
        "IRRADIATION",
        "AMBIENT_TEMPERATURE",
        "MODULE_TEMPERATURE",
        "AC_POWER"
    ]
].corr()

plant2_corr = plant2_analysis[
    [
        "IRRADIATION",
        "AMBIENT_TEMPERATURE",
        "MODULE_TEMPERATURE",
        "AC_POWER"
    ]
].corr()

print("Plant 1 Correlation:")
display(plant1_corr)

print("\nPlant 2 Correlation:")
display(plant2_corr)

In [ ]:
# Step 19: Compare average inverter-level AC power
# This identifies differences in inverter performance.

plant1_inverter = (
    plant1_generation
    .groupby("SOURCE_KEY")["AC_POWER"]
    .mean()
    .sort_values(ascending=False)
)

plant2_inverter = (
    plant2_generation
    .groupby("SOURCE_KEY")["AC_POWER"]
    .mean()
    .sort_values(ascending=False)
)

print("Plant 1 Inverter Performance:")
print(plant1_inverter)

print("\nPlant 2 Inverter Performance:")
print(plant2_inverter)

In [ ]:
# Plot inverter-level average AC power

plt.figure(figsize=(12, 5))

plant1_inverter.plot(kind="bar")

plt.xlabel("Inverter")
plt.ylabel("Average AC Power")
plt.title("Plant 1 - Inverter Performance")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Step 20: Identify inverters with relatively low average output

plant1_threshold = plant1_inverter.quantile(0.10)
plant2_threshold = plant2_inverter.quantile(0.10)

print("Plant 1 Low-Performance Threshold:", plant1_threshold)
print("Plant 2 Low-Performance Threshold:", plant2_threshold)

print("\nPotentially Low-Performing Plant 1 Inverters:")
print(plant1_inverter[plant1_inverter <= plant1_threshold])

print("\nPotentially Low-Performing Plant 2 Inverters:")
print(plant2_inverter[plant2_inverter <= plant2_threshold])

In [ ]:
# Final check for missing values in the analysis datasets

for name, df in {
    "Plant 1 Analysis": plant1_analysis,
    "Plant 2 Analysis": plant2_analysis
}.items():

    print(name)
    print(df.isnull().sum())
    print()

In [ ]:
# Check correlations between numerical variables and AC power

print("Plant 1 Correlation:")
display(
    plant1_analysis[
        ["IRRADIATION", "AMBIENT_TEMPERATURE",
         "MODULE_TEMPERATURE", "AC_POWER"]
    ].corr()
)

print("\nPlant 2 Correlation:")
display(
    plant2_analysis[
        ["IRRADIATION", "AMBIENT_TEMPERATURE",
         "MODULE_TEMPERATURE", "AC_POWER"]
    ].corr()
)

In [ ]:
# Analyze daily energy yield over time

plant1_yield = (
    plant1_generation
    .groupby(plant1_generation["DATE_TIME"].dt.date)["DAILY_YIELD"]
    .max()
)

plant2_yield = (
    plant2_generation
    .groupby(plant2_generation["DATE_TIME"].dt.date)["DAILY_YIELD"]
    .max()
)

plt.figure(figsize=(12, 5))

plt.plot(plant1_yield.index, plant1_yield.values, label="Plant 1")
plt.plot(plant2_yield.index, plant2_yield.values, label="Plant 2")

plt.xlabel("Date")
plt.ylabel("Daily Yield")
plt.title("Daily Solar Energy Yield")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.show()